In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- 1. ADIM: Veriyi Çekme ---
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# --- 2. ADIM: Temizlik ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop('customerID', axis=1, inplace=True)

# --- 3. ADIM: ÖZELLİK MÜHENDİSLİĞİ (MODELİ ZEKİLEŞTİREN KISIM) ---
# Müşterinin kaç aydır bizde olduğunu, aylık faturasına bölerek bir "Bağlılık/Maliyet" oranı çıkarıyoruz.
df['Tenure_to_MonthlyCharges'] = df['tenure'] / df['MonthlyCharges']

# Toplamda kaç farklı ekstra hizmet (Güvenlik, TV, Destek vb.) aldığını sayan yeni bir sütun üretiyoruz.
hizmetler = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
             'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Total_Services_Count'] = (df[hizmetler] != 'No').sum(axis=1)

# --- 4. ADIM: Hedef (y) ve Özellik (X) Ayrımı ---
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_encoded = pd.get_dummies(X, drop_first=True)

# --- 5. ADIM: Veriyi Bölme (80-10-10) ---
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# DİKKAT: Accuracy'yi maksimize etmek için SMOTE'u kapattık!

# --- 6. ADIM: YENİ MODEL - XGBoost ---
xgb_model = XGBClassifier(
    random_state=42,
    learning_rate=0.03,    # Daha da kısık ateşte yavaş yavaş öğreniyor
    max_depth=5,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_val)

# --- SONUÇLAR ---
print("\n--- XGBOOST (FEATURE ENGINEERING) BAŞARI KARNESİ ---")
print(classification_report(y_val, y_pred))

accuracy = accuracy_score(y_val, y_pred) * 100
print(f"\nGenel Doğruluk Oranı (Accuracy): %{accuracy:.2f}")


--- XGBOOST (FEATURE ENGINEERING) BAŞARI KARNESİ ---
              precision    recall  f1-score   support

           0       0.83      0.88      0.86       516
           1       0.61      0.52      0.56       187

    accuracy                           0.79       703
   macro avg       0.72      0.70      0.71       703
weighted avg       0.78      0.79      0.78       703


Genel Doğruluk Oranı (Accuracy): %78.52
